In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M05.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7799628743569516, 'n_it': 0.37514248216589685}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 1000

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[15.855823255435215, 13.707338029338883, 13.872389408464448, 13.656042371697048, 14.405797647958511, 13.578247513101644, 17.466689926447103, 14.766078490627066, 15.258053712099379, 14.735578055403554, 13.889360277040943, 14.319991986863942, 14.499541227523853, 13.759011636549388, 14.224813012726486, 13.794017284087662, 13.797768371810715, 16.72082809416761, 16.4512105925717, 14.908116982694201, 16.58912463817577, 15.991471500342325, 15.317487294493265, 13.689591681654267, 13.59317648525025, 16.143615222857704, 15.063329179228724, 16.606886570264226, 13.349679821408266, 13.457797831861548, 14.501314385538288, 15.928737195970244, 15.091406227509655, 13.625416491813862, 14.647293467399875, 15.494344424689293, 15.86223461499305, 14.096356423351835, 16.908644773801882, 16.403218902049886, 13.456622313522766, 13.822130721233211, 14.826105075013887, 13.674145561716188, 13.788927259194791, 17.378422853393403, 16.328199724624263, 15.251563664629629, 15.012926643371198, 14.015541625890735, 14.93

In [5]:
np.average(y_max_arr)

np.float64(14.832439929426611)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M05/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)